# Regrouper les conduites autrement

Le carnet 2 s'est terminé sur un résultat négatif : les six coefficients de rugosité changent de
valeur selon la fenêtre sur laquelle on les règle, donc ils ne mesurent rien.

Ce carnet pose la question qui manquait : **est-ce le paramètre qui n'est pas identifiable, ou le
regroupement qui est mauvais ?**

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from calibration import reseau as R, profils as P, rugosite as U, ameliorations as A, diagnostics as G

plt.rcParams.update({"figure.figsize": (10, 3.2), "figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 9})

ANNEE = 2018
N_ENTR = 2 * R.PAS_JOUR          # fenêtre d'ajustement
N_PAS = 7 * R.PAS_JOUR           # fenêtre d'évaluation
FEN = {"semaine 1": 0, "janvier": 3 * R.PAS_JOUR, "juillet": 181 * R.PAS_JOUR,
       "octobre": 275 * R.PAS_JOUR}

R.verifier_donnees(ANNEE)
wn = R.charger_modele(duree_h=24)
noeuds = wn.junction_name_list
print("prêt")

prêt


## 1. Ce que le fichier contient vraiment

Avant de regrouper, regardons les valeurs à regrouper.

In [2]:
rugosites = pd.Series({p: wn.get_link(p).roughness for p in wn.pipe_name_list})
diametres = pd.Series({p: round(wn.get_link(p).diameter * 1000) for p in wn.pipe_name_list})

print("coefficients de Hazen-Williams présents dans le fichier :")
print(rugosites.value_counts().to_string())

coefficients de Hazen-Williams présents dans le fichier :
140.0    786
120.0    119


**Deux valeurs, et deux seulement.** C'est déjà une information : le jeu de données a été
construit en partant de deux familles de conduites, puis en perturbant les paramètres.

Regardons comment ces deux valeurs se répartissent dans les groupes par diamètre.

In [3]:
croise = pd.crosstab(diametres, rugosites)
croise.index.name = "diamètre (mm)"; croise.columns.name = "coefficient"
print(croise.to_string())

coefficient    120.0  140.0
diamètre (mm)              
63                 2      2
75                 0      1
100              104    601
150               13     90
160                0     16
200                0     64
225                0     12


### Le défaut

Le diamètre 100 mm, à lui seul 705 conduites sur 905, contient **les deux** coefficients : 104
conduites à 120 et 601 à 140. Le groupe `D100` du carnet 1 leur impose donc **un seul** paramètre.

Un paramètre unique pour deux valeurs vraies différentes ne peut pas être stable : selon ce que la
fenêtre d'ajustement met en avant, il penchera d'un côté ou de l'autre. C'est peut-être tout ce
qu'on observait au carnet 2.

## 2. Un regroupement qui respecte ce qu'on sait

Trois critères, tous lisibles dans le fichier sans rien estimer :

* le **coefficient de départ**, pour ne pas mélanger deux valeurs vraies ;
* le **diamètre**, parce que c'est lui qui pèse dans la résistance ;
* la **zone**, C ou A+B, parce que les conditions hydrauliques n'y sont pas les mêmes — la zone C
  est derrière le réservoir.

C'est `critere="physique"` dans `reseau.groupes_de_rugosite`.

In [4]:
par_diametre = R.groupes_de_rugosite(wn, n_groupes=6)
physique = R.groupes_de_rugosite(wn, critere="physique")

print(f"par diamètre : {len(set(par_diametre.values()))} groupes")
print(f"physique     : {len(set(physique.values()))} groupes\n")
print(pd.Series(physique).value_counts().to_string())

par diamètre : 6 groupes
physique     : 13 groupes

AB|C140|D100           497
AB|C120|D100           104
C|C140|D100            104
AB|C140|D150            90
AB|C140|D200            57
AB|C140|D160            16
AB|C120|D150            13
AB|C140|D225            12
C|C140|D200              4
frontiere|C140|D200      3
AB|C140|D63              2
AB|C120|D63              2
AB|C140|D75              1


Treize groupes au lieu de six. C'est plus de paramètres sur un réseau qui en supportait déjà mal
six — donc rien ne dit d'avance que ce soit mieux. Il faut mesurer.

## 3. Le même test qu'au carnet 2

On refait **le même ajustement sur trois fenêtres**, dont deux sans aucune fuite (la semaine 1, et
janvier trois jours plus tard). Si un regroupement est bon, ses coefficients ne doivent pas
dépendre de la fenêtre.

In [5]:
# Le modèle calibré du carnet 2, repris tel quel.
formes = P.formes_par_categorie(wn, ANNEE)
D, _ = P.demandes_calibrees(wn, ANNEE, formes)
D = A.recaler_zone_ab(D, noeuds, wn, ANNEE, periode=R.PAS_SEMAINE)
statut = A.statut_pompe(ANNEE)
niveau = R.charger_niveau(ANNEE).to_numpy()
pres = R.charger_pressions(ANNEE).to_numpy()
print("prêt")

prêt


In [6]:
def simuler(groupes, debut, n, coefs=None):
    return R.pressions_simulees(
        D[debut:debut + n + 1], noeuds, n, tranche_jours=1, niveau0=float(niveau[debut]),
        verbeux=False, pompe=statut[debut:debut + n + 1],
        niveau_mesure=niveau[debut:debut + n + 1],
        coefficients=None if coefs is None else U.coefficients_par_conduite(groupes, coefs))

def ajuster_sur(groupes, debut):
    depart = U.coefficients_initiaux(wn, groupes)
    return U.ajuster(lambda c: simuler(groupes, debut, N_ENTR, c),
                     pres[debut:debut + N_ENTR], depart, max_nfev=12, verbeux=False)

In [7]:
# Dispersion sans aucun ajustement : le repère auquel tout se compare.
reference = {nom: G.resume(simuler(par_diametre, d, N_PAS), pres[d:d + N_PAS])["dispersion_m"]
             for nom, d in FEN.items()}
print({k: round(v, 4) for k, v in reference.items()})

{'semaine 1': 0.0833, 'janvier': 0.0803, 'juillet': 0.1882, 'octobre': 0.2141}


In [8]:
# Six ajustements : trois fenêtres × deux regroupements. Comptez une dizaine de minutes.
solutions = {}
for lib, groupes in [("par diamètre", par_diametre), ("physique", physique)]:
    for fen in ("semaine 1", "janvier", "juillet"):
        s = ajuster_sur(groupes, FEN[fen])
        solutions[(lib, fen)] = s
        print(f"{lib:13s} réglé sur {fen:10s} : {s['n_evaluations']:3d} simulations, "
              f"RMSE {s['rmse_depart']:.4f} → {s['rmse']:.4f}", flush=True)

par diamètre  réglé sur semaine 1  :  67 simulations, RMSE 0.1174 → 0.0734


par diamètre  réglé sur janvier    :  61 simulations, RMSE 0.0948 → 0.0719


par diamètre  réglé sur juillet    :  73 simulations, RMSE 0.2376 → 0.1412


physique      réglé sur semaine 1  : 143 simulations, RMSE 0.1220 → 0.0630


physique      réglé sur janvier    : 104 simulations, RMSE 0.0988 → 0.0712


physique      réglé sur juillet    : 130 simulations, RMSE 0.2448 → 0.1201


## 4. Les coefficients sont-ils stables ?

C'est la seule question. Un coefficient de rugosité décrit une conduite ; il ne doit pas dépendre
de la semaine pendant laquelle on l'a regardé.

In [9]:
for lib, groupes in [("par diamètre", par_diametre), ("physique", physique)]:
    depart = U.coefficients_initiaux(wn, groupes)
    t = pd.DataFrame({"conduites": pd.Series(groupes).value_counts(), "fichier": pd.Series(depart),
                      **{f: pd.Series(solutions[(lib, f)]["coefficients"])
                         for f in ("semaine 1", "janvier", "juillet")}})
    t = t.sort_values("conduites", ascending=False)
    print(f"\n=== {lib} ===")
    print(t.round(0).to_string())


=== par diamètre ===
       conduites  fichier  semaine 1  janvier  juillet
D100         705    137.0       82.0    137.0     72.0
D150         103    138.0      160.0    137.0    160.0
D200          64    140.0      146.0    128.0    150.0
D160          16    140.0      160.0     75.0    160.0
D225          12    140.0      160.0     76.0    160.0
D<=75          5    135.0      160.0    109.0    160.0

=== physique ===
                     conduites  fichier  semaine 1  janvier  juillet
AB|C140|D100               497    140.0      130.0    130.0    123.0
AB|C120|D100               104    120.0      105.0    108.0     90.0
C|C140|D100                104    140.0       79.0    145.0     70.0
AB|C140|D150                90    140.0      142.0    145.0    149.0
AB|C140|D200                57    140.0      128.0    128.0    125.0
AB|C140|D160                16    140.0      156.0    150.0    108.0
AB|C120|D150                13    120.0      110.0    103.0    114.0
AB|C140|D225           

In [10]:
# Un seul chiffre : de combien un coefficient bouge d'une fenêtre à l'autre,
# pondéré par le nombre de conduites que le groupe porte.
def instabilite(lib, groupes):
    n = pd.Series(groupes).value_counts()
    v = pd.DataFrame({f: pd.Series(solutions[(lib, f)]["coefficients"])
                      for f in ("semaine 1", "janvier", "juillet")})
    etendue = v.max(axis=1) - v.min(axis=1)                 # écart max-min par groupe
    return float((etendue * n).sum() / n.sum())

for lib, groupes in [("par diamètre", par_diametre), ("physique", physique)]:
    print(f"{lib:13s} : écart max-min moyen entre fenêtres, par conduite = "
          f"{instabilite(lib, groupes):5.1f} unités de Hazen-Williams")

par diamètre  : écart max-min moyen entre fenêtres, par conduite =  58.0 unités de Hazen-Williams
physique      : écart max-min moyen entre fenêtres, par conduite =  17.5 unités de Hazen-Williams


### Lecture

L'instabilité est divisée par plus de trois : **58** unités d'écart entre fenêtres avec les
groupes par diamètre, **17,5** avec les groupes physiques.

Le détail est plus parlant que le résumé. Les quatre plus gros groupes de la zone A+B —
748 conduites sur 905 — tombent maintenant à quelques unités près d'une fenêtre à l'autre, et près
de leur valeur de fichier :

| groupe | conduites | fichier | semaine 1 | janvier | juillet |
|---|---|---|---|---|---|
| AB / C140 / D100 | 497 | 140 | 130 | 130 | 123 |
| AB / C120 / D100 | 104 | 120 | 105 | 108 | 90 |
| AB / C140 / D150 | 90 | 140 | 142 | 145 | 149 |
| AB / C140 / D200 | 57 | 140 | 128 | 128 | 125 |

Là où le groupe `D100` unique donnait 82, 137 puis 72 selon la fenêtre.

**La conclusion du carnet 2 était donc trop large.** Ce n'est pas la rugosité qui était
inidentifiable : c'est qu'un groupe mélangeant deux valeurs vraies ne peut pas converger, et que
ce groupe couvrait 78 % du réseau.

## 5. Et est-ce que ça transfère mieux ?

La stabilité ne suffit pas : des coefficients stables mais faux seraient stables. On regarde donc
aussi ce que chaque réglage donne sur des fenêtres qu'il n'a jamais vues.

In [11]:
def transfert(groupes, coefs):
    return {nom: 100 * (G.resume(simuler(groupes, d, N_PAS, coefs),
                                 pres[d:d + N_PAS])["dispersion_m"] / reference[nom] - 1)
            for nom, d in FEN.items()}

for lib, groupes in [("par diamètre", par_diametre), ("physique", physique)]:
    t = pd.DataFrame({f: transfert(groupes, solutions[(lib, f)]["coefficients"])
                      for f in ("semaine 1", "janvier", "juillet")})
    print(f"\n=== {lib} — variation de dispersion (%), lignes = évaluation, colonnes = réglage ===")
    print(t.round(1).to_string())


=== par diamètre — variation de dispersion (%), lignes = évaluation, colonnes = réglage ===
           semaine 1  janvier  juillet
semaine 1       -3.1    -10.6     10.0
janvier          8.9    -16.6     28.1
juillet        -13.8      4.6    -13.2
octobre         -4.1      8.7     -3.0



=== physique — variation de dispersion (%), lignes = évaluation, colonnes = réglage ===
           semaine 1  janvier  juillet
semaine 1        1.4    -10.1     13.7
janvier         21.1    -17.7     38.9
juillet        -20.4      6.7    -21.2
octobre         -7.0      9.9     -7.2


### Lecture

Le transfert, lui, n'est pas meilleur : il est **plus extrême dans les deux sens**. Réglé sur la
semaine 1, le regroupement physique gagne 20 % en juillet au lieu de 14, et 7 % en octobre au lieu
de 4 — mais il dégrade janvier de 21 % au lieu de 9.

Treize paramètres au lieu de six, c'est plus de liberté, donc plus de capacité à courir après sa
propre fenêtre. Reste à savoir où cette liberté est passée.

## 6. Où est passée cette liberté ?

Les groupes qui s'arrêtent sur une borne sont toujours les mêmes, et ce sont les plus petits.
Regardons quelles conduites ils contiennent.

In [12]:
for nom in ("frontiere|C140|D200", "C|C140|D200"):
    conduites = [p for p, g in physique.items() if g == nom]
    print(f"{nom} ({len(conduites)} conduites) :")
    for p in conduites:
        lien = wn.get_link(p)
        print(f"   {p:6s} {lien.start_node_name:>5s} → {lien.end_node_name:<5s} {lien.length:6.1f} m")

frontiere|C140|D200 (3 conduites) :
   p227      R1 → n303    26.9 m
   p235      R2 → n336    36.2 m
   p239      T1 → n343    27.0 m
C|C140|D200 (4 conduites) :
   p240    n343 → n44     24.0 m
   p316     n44 → n387    48.3 m
   p317    n387 → n388    48.3 m
   p318    n388 → n45     48.3 m


### Lecture

Ce ne sont pas des conduites quelconques. `p227` et `p235` sont les **deux entrées du réseau**,
`p239` la **sortie du réservoir**, et `p240` à `p318` la conduite de transport qui part du
réservoir vers la zone C.

Sept conduites — et tout le débit du réseau passe par elles.

Le regroupement physique a donc fait quelque chose de plus utile que stabiliser des coefficients :
il a **isolé le problème**. La liberté que l'optimiseur exploite ne se disperse plus sur
705 conduites anonymes, elle se concentre sur sept conduites qu'on peut nommer — et qui sont
précisément celles dont le carnet 2 a montré que le débit dépend de l'état de la pompe et du
niveau du réservoir.

Ces sept coefficients ne mesurent donc pas de la friction : ils servent de bouton de réglage pour
la condition aux limites.

## Ce qu'il faut retenir

1. **Regrouper par diamètre seul était une erreur**, et une erreur mesurable. Le groupe `D100`
   réunissait 104 conduites de coefficient vrai 120 et 601 de coefficient 140 : un paramètre pour
   deux valeurs, donc un paramètre qui oscille avec la fenêtre.
2. **Croiser zone × coefficient × diamètre divise l'instabilité par plus de trois** — 58 unités
   d'écart entre fenêtres contre 17,5 — et rend stables les quatre groupes qui portent
   748 conduites sur 905.
3. **La conclusion du carnet 2 doit donc être corrigée.** La rugosité n'est pas « inidentifiable
   sur ce réseau » : elle l'est pour l'essentiel du réseau, à condition de ne pas mélanger deux
   valeurs vraies dans un même groupe.
4. **Elle ne rapporte pourtant toujours rien**, et pour une raison désormais plus nette : les
   coefficients identifiables ne s'écartent que de 5 à 10 % de leur valeur de fichier — le fichier
   avait déjà à peu près raison — tandis que ce qui bouge vraiment se concentre sur **sept
   conduites** qui servent de bouton de réglage pour la condition aux limites.
5. **Un mauvais regroupement se diagnostique avant l'optimisation.** Il suffit de compter combien
   de valeurs distinctes chaque groupe contient. Ici, une ligne de code aurait suffi, et elle
   aurait évité de conclure trop large.